# utils

> Tacit BasedPL primitives for strings, lines, HTML regex extraction, paths, JSON I/O, rate-limited `•fetch`, and answer coercion

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.all import *

In [ ]:
#| export
import contextlib
from decimal import Decimal
from fractions import Fraction
from functools import reduce as py_reduce
from importlib.metadata import version
from basedpl import *

## Exceptions

Minimal error hierarchy for missing tokens, locked puzzles, and invalid submissions.

In [ ]:
#| export
class AocdError(Exception): "Base exception for baocd."
class PuzzleLockedError(AocdError): "Puzzle input is not available yet."
class DeadTokenError(AocdError): "Session token is expired or invalid."

## Tacit String, Line, & Record Combinators

Pure tacit BasedPL `Function` variables built with `atop`, `fork`, `|`, `.before`, `.each`, and `.agenda`.

In [ ]:
#| export
bstrip = replace | Array([r"^\s+|\s+$", ""])
brstrip_nl = replace | Array([r"[\r\n]+$", ""])
_strip_lead_slash = replace | Array([r"^/+", ""])
_strip_trail_slash = replace | Array([r"/+$", ""])
_lower = -1 | c

bhas = atop(or_.reduce, find)
bstartswith = fork(same_left, match, tally.before(take))
bendswith = fork(same_left, match, (-tally).before(take))
bwords = atop(not_, member | Array(" \t\r\n")).before(partition)
bsplit_path = atop(not_, member | Array("/\\")).before(partition)

_bjoin_ne = fork(atop(tally, same_left), drop, atop(enlist, fork(atop(enclose, same_left), catenate.each, same)))
bjoin = atop(0 | less, tally, same).agenda(Array([Array(""), _bjoin_ne]))

_is_nonempty_str = fork(atop(0 | less, tally), and_, atop(1 | equal, depth))
_split_ne = atop(
    (1 | drop).each,
    (equal | Array("\n")).before(partitioned_enclose),
    Array("\n") | catenate,
    without | Array("\r"),
)
_split_str = atop(atop(0 | less, tally).agenda(Array([json("[]"), _split_ne])), brstrip_nl)
bsplit_lines = _is_nonempty_str.agenda(Array([atop(0 | less, tally).agenda(Array([json("[]"), same])), _split_str]))

Small Python-to-BPL interop helpers, keyed record accessors, and shared `test_beq`.

In [ ]:
#| export
def bstr(x):
    "Ensure `x` is a BasedPL character vector `Array`."
    if isinstance(x, Array): return x if x._inner.text is not None else Array(str(x.py))
    return Array(str(x))


def pystr(x):
    "Extract a Python `str` from a BasedPL `Array` or Python value."
    if isinstance(x, Array): return x.py if x._inner.text is not None else str(x.py)
    return str(x)


def aiter(arr):
    "Iterate disclosed items of a 1D BasedPL `Array`."
    for i in range(len(arr)): yield arr[i]


def beq(a, b):
    "Structural equality via BasedPL `match` (`≡`), safely handling `None`."
    if a is None or b is None: return a is None and b is None
    return bool(match(a if isinstance(a, Array) else Array(a), b if isinstance(b, Array) else Array(b)))


def test_beq(a, b):
    "Test BasedPL structural equality (`≡`) using `fastcore.all.test`."
    from fastcore.all import test
    test(a, b, beq, cname="≡")


def rec_keys(rec): return rec.keys[0] if rec.keys and rec.keys[0] is not None else ()
def rec_has(rec, key): return key in rec_keys(rec)


def rec_set(rec, key, val):
    "Return keyed BasedPL `Array` `rec` with `key` set or appended to `val`."
    if not rec_has(rec, key): return catenate(rec, Array({key: val}))
    out = Array(rec)
    out[key] = val
    return out

In [ ]:
test_beq(bstrip("  hello \n "), Array("hello"))
test_beq(bsplit_lines("L68\nL30\n\nR48\n"), Array(["L68", "L30", "", "R48"]))
test_beq(bjoin(", ", Array(["aa", "bb", "cc"])), Array("aa, bb, cc"))

## Pure Tacit HTML Extraction & Entity Decoding

Replaces a full DOM parser with tacit BasedPL `•r`, `•replace`, `16⊥`, and `•ucs`.

In [ ]:
#| export
_HEX = Array("0123456789abcdef")
_NAMED_ENTITIES = (
    ("&lt;", "<"), ("&gt;", ">"), ("&quot;", '"'), ("&apos;", "'"), ("&#39;", "'"),
    ("&nbsp;", "\u00a0"), ("&mdash;", "\u2014"), ("&ndash;", "\u2013"), ("&hellip;", "\u2026"),
    ("&amp;", "&"),
)
_dec_char = atop(ravel, ucs, vfi[1][0])
_hex_char = atop(ravel, ucs, 16 | decode, _HEX | index_of, _lower)
_unescape_named = py_reduce(lambda f, kv: atop(replace | Array(list(kv)), f), _NAMED_ENTITIES, same)
strip_tags = replace | Array([r"(?s)<[^>]+>", ""])


def _decode_entities(s, pat, to_char):
    m = r(s, pat)
    if not len(m.text): return s
    um = unique_mask(m.text)
    ks, gs = replicate(um, m.text), replicate(um, m.groups)
    for i in range(len(ks)): s = replace(s, catenate(enclose(ks[i]), enclose(to_char(gs[i][0]))))
    return s


def unescape_html(text):
    "Decode numeric and named HTML entities natively on BasedPL `Array`s."
    s = bstr(text)
    if not len(s): return s
    s = _decode_entities(s, r"&#([0-9]+);", _dec_char)
    s = _decode_entities(s, r"&#[xX]([0-9a-fA-F]+);", _hex_char)
    return _unescape_named(s)


def tag_texts(html, tag, strip_ws=False):
    "Extract inner text of every `<tag>...</tag>` element as a list of unescaped BasedPL `Array`s."
    gs = r(bstr(html), rf"(?is)<{tag}\b[^>]*>(.*?)</{tag}>").groups
    out = [unescape_html(strip_tags(gs[i][0])) for i in range(len(gs))]
    return [bstrip(x) for x in out] if strip_ws else out

In [ ]:
test_beq(unescape_html("a &lt; b &amp; &#65; &#x42;"), Array("a < b & A B"))
test_beq(tag_texts("<article><pre><code>L68\n&lt;x&gt;</code></pre></article>", "pre")[0], Array("L68\n<x>"))

## Tacit Paths, Filesystem, & JSON I/O

Tacit `expand_path`, `parent_dir`, `file_name`, `path_kind`, `is_dir`, and `is_file` with keyword-arg `•nput`/`•nget`.

In [ ]:
#| export
_host_env = atop(host, Array("env") | same_left)
_pick_empty = pick.axis(Array(fill=Array("")))
_get_home = atop(
    fork(
        Array("HOME") | _pick_empty,
        atop(0 | less, tally, same_left).agenda(Array([same, same_left])),
        Array("USERPROFILE") | _pick_empty,
    ),
    _host_env,
)

expand_path = fork(Array("~") | match, or_, Array("~/") | bstartswith).agenda(Array([same, fork(_get_home, catenate, 1 | drop)]))
parent_dir = atop(path["parent"], expand_path)
file_name = atop(path["name"], expand_path)
path_kind = atop(metadata["kind"][0], expand_path)
is_dir = atop(Array("dir") | match, path_kind)
is_file = atop(member | Array(["file", "symlink"]), enclose, path_kind)


def env_get(name, default=None):
    "Read an environment variable via `•host 'env'`."
    env = host("env")
    if default is not None: return pick.axis(Array(fill=bstr(default)))(name, env)
    return env[name] if rec_has(env, name) else None


def join_path(*parts):
    "Join path segments into a clean BasedPL path `Array`."
    segs = []
    for i, part in enumerate(parts):
        s = bstr(part)
        if not len(s): continue
        if i > 0: s = _strip_lead_slash(s)
        s = _strip_trail_slash(s)
        if len(s): segs = [*segs, s]
        elif i == 0 and bool(bstartswith("/", bstr(part))): segs = [*segs, Array("")]
    if not segs: return Array("")
    res = bjoin("/", Array(segs)) if len(segs) > 1 else segs[0]
    return res if len(res) else Array("/")


def read_text(p):
    "Read a UTF-8 file via `•nget`."
    ep = expand_path(p)
    if not is_file(ep): raise FileNotFoundError(pystr(ep))
    return nget(ep)


def write_text(p, contents):
    "Write UTF-8 text via `•mkdir` and `•nput` (`overwrite=1`)."
    ep = expand_path(p)
    par = parent_dir(ep)
    if len(par) and not is_dir(par): mkdir(par)
    nput(bstr(contents), path=ep, overwrite=1)


def atomic_write_file(p, contents):
    "Atomically write text via unique `•nput` and `•rename`."
    ep = expand_path(p)
    par = parent_dir(ep)
    if len(par) and not is_dir(par): mkdir(par)
    tmp = nput(bstr(contents), path=par if len(par) else Array("."), unique=1, prefix=".baocd_tmp_")
    rename(ep, tmp)


def read_json(p): return json(read_text(p))
def write_json(p, obj): atomic_write_file(p, json.undo(obj if isinstance(obj, Array) else Array(obj)))

## Functional Rate-Limited HTTP (`•fetch`) & Answer Coercion

Pure tacit `percent_encode`, functional `http_get`/`http_post` returning `•fetch`'s native `Array`, and `coerce`.

In [ ]:
#| export
try: _pkg_ver = version("baocd")
except Exception: _pkg_ver = "0.1.0"
_BASE_HEADERS = Array({"User-Agent": f"github.com/AnswerDotAI/baocd baocd/{_pkg_ver}"})

_HEX_UPPER = Array("0123456789ABCDEF")
_UNRESERVED_BYTES = ucs("UTF-8", enlist(Array([bpl("$a"), c(-1, bpl("$a")), bpl("$d"), Array("-_.~")])))
_hex_byte = atop(enclose, Array("%") | catenate, _HEX_UPPER | index_of.undo, Array([16, 16]) | encode).rank(0)
_sel_byte = atop(double, not_, member | _UNRESERVED_BYTES) - (equal | 32)
percent_encode = atop(0 | less, tally).agenda(Array([
    Array(""),
    atop(enlist, _sel_byte.agenda(Array([atop(enclose, ucs).rank(0), enclose(Array("+")), _hex_byte])), Array("UTF-8") | ucs),
]))
_pair_eq = fork(atop(percent_encode, same_left), catenate, atop(Array("=") | catenate, percent_encode, same))

_http_max_t, _http_cooloff = 3.0, 0.16
_http_history = reshape(Array([4]), time(0) - _http_max_t)


def form_encode(fields):
    "Encode a keyed `Array` or dict as `application/x-www-form-urlencoded`."
    items = [(k, fields[k]) for k in rec_keys(fields)] if isinstance(fields, Array) else list(fields.items())
    if not items: return Array("")
    pairs = [_pair_eq(bstr(k), bstr(v)) for k, v in items]
    return pairs[0] if len(pairs) == 1 else bjoin("&", Array(pairs))


def _rate_limit():
    global _http_cooloff, _http_history
    now = float(time(0))
    if now - float(_http_history[0]) < _http_max_t:
        delay(_http_cooloff)
        _http_cooloff = float(minimum(Array(10.0), double(Array(_http_cooloff))))
    _http_history = catenate(drop(1, _http_history), Array(now))


def _headers(token=None, extra=None):
    hdrs = _BASE_HEADERS
    if token is not None: hdrs = rec_set(hdrs, "Cookie", catenate(Array("session="), bstr(token)))
    if extra:
        for k, v in extra.items(): hdrs = rec_set(hdrs, k, bstr(v))
    return hdrs


def http_get(url, token=None):
    "Rate-limited HTTP GET via `•fetch`, returning a native BasedPL `Array` (`status`, `headers`, `body`)."
    _rate_limit()
    return fetch(bstr(url), headers=_headers(token=token))


def http_post(url, token, fields):
    "Rate-limited HTTP POST via `•fetch`, returning a native BasedPL `Array` (`status`, `headers`, `body`)."
    _rate_limit()
    return fetch(bstr(url), headers=_headers(token=token, extra={"Content-Type": "application/x-www-form-urlencoded"}), body=form_encode(fields))


def coerce(val):
    "Convert an answer (BPL Array, int, integral float/complex/Decimal/Fraction, bytes, str) to a BasedPL string `Array`."
    orig = val
    if isinstance(val, Array):
        if val._inner.text is not None: return val
        if not val.shape or (int(tally(val)) == 1 and all(d == 1 for d in val.shape)): val = val._atom()
        else: raise AocdError(f"Failed to coerce {type(orig).__name__} value {orig!r} to str")
    with contextlib.suppress(ImportError):
        import numpy as np
        if isinstance(val, np.ndarray) and val.size == 1: val = val.item()
        if isinstance(val, (np.integer, np.floating, np.complexfloating)) and val.imag == 0 and float(val.real).is_integer():
            val = str(int(val.real))
    if isinstance(val, bool): raise AocdError(f"Failed to coerce bool value {orig!r} to str")
    if isinstance(val, int): val = str(val)
    elif isinstance(val, (float, complex)) and val.imag == 0 and float(val.real).is_integer(): val = str(int(val.real))
    elif isinstance(val, bytes): val = val.decode()
    elif isinstance(val, (Decimal, Fraction)):
        num, den = val.as_integer_ratio()
        if den == 1: val = str(num)
    if not isinstance(val, str): raise AocdError(f"Failed to coerce {type(orig).__name__} value {orig!r} to str")
    return Array(val)

In [ ]:
test_beq(form_encode(Array(level="1", answer="hello world+1")), Array("level=1&answer=hello+world%2B1"))
test_beq(coerce(Array(42)), Array("42"))
test_fail(lambda: coerce(True), AocdError)